# AE 370 Group 21: Atmospheric Entry Simulation Parameters
This notebook defines all physical constants, vehicle properties, and initial conditions (IVP) required to solve the equations of motion for a spacecraft returning from deep space.


In [13]:
from sympy import *
from scipy.integrate import solve_ivp
import numpy as np
import matplotlib.pyplot as plt

In [14]:
# --- Physical Constants ---
R_E = 6378137.0         # Radius of Earth (meters)
g0 = 9.80665            # Standard gravity at sea level (m/s^2)

# --- Atmospheric & Aerodynamic Assumptions ---
rho_0 = 1.225           # Sea-level atmospheric density (kg/m^3)
H_scale = 7500.0        # Atmospheric scale height (meters)
mass = 5000.0           # Spacecraft mass (kg)
area = 10.0             # Reference frontal area (m^2)
C_D = 1.5               # Baseline drag coefficient
C_L = 0.5               # Baseline lift coefficient



In [15]:
def get_density(h):
    """Calculates atmospheric density using an exponential model assumption."""
    return rho_0 * np.exp(-h / H_scale) if h >= 0 else rho_0

def calc_forces_and_load(h, v):
    """Calculates lift, drag, and the total load factor in Earth g's."""
    rho = get_density(h)
    q = 0.5 * rho * (v**2)  # Dynamic pressure
    
    lift = q * C_L * area
    drag = q * C_D * area
    load_g = np.sqrt(lift**2 + drag**2) / (mass * g0)
    
    return lift, drag, load_g


In [16]:
def entry_equations_of_motion(t, state):
    """
    Defines the 4 ODE derivatives for the state vector: [h, v, gamma, s]
    where s is the downrange distance traveled over the Earth's surface.
    """
    h, v, gamma, s = state
    
    # Fetch forces
    lift, drag, _ = calc_forces_and_load(h, v)
    g = g0 * (R_E / (R_E + h))**2  # Gravity variation with altitude
    
    # 1. Altitude derivative
    dh_dt = v * np.sin(gamma)
    
    # 2. Velocity derivative
    dv_dt = -(drag / mass) - g * np.sin(gamma)
    
    # 3. Flight-path angle derivative
    dgamma_dt = (lift / (mass * v)) + ((v / (R_E + h)) - (g / v)) * np.cos(gamma)
    
    # 4. Downrange distance derivative (The 4th State)
    ds_dt = (R_E / (R_E + h)) * v * np.cos(gamma)
    
    return [dh_dt, dv_dt, dgamma_dt, ds_dt]


In [17]:
# --- Initial Value Problem (IVP) Conditions ---
h_0 = 120000.0          # Entry interface altitude (120 km)
v_0 = 11200.0           # Escape velocity entry speed (m/s)
gamma_0 = np.radians(-6.0) # Entry angle converted to radians
s_0 = 0.0               # Initial downrange distance (meters)

initial_state = [h_0, v_0, gamma_0, s_0]
t_span = (0, 2000)      # Max simulation window (seconds)


In [18]:
def crash_event(t, state):
    """Triggers if the spacecraft hits the surface (altitude <= 0)."""
    return state[0]  # returns altitude h
crash_event.terminal = True
crash_event.direction = -1

def skip_event(t, state):
    """Triggers if the spacecraft bounces back out into deep space (h >= 150km after dipping)."""
    # Only active if time has progressed past the initial entry second
    return state[0] - 150000.0 if t > 10 else 1.0
skip_event.terminal = True
skip_event.direction = 1

# List of tracking thresholds to pass to scipy's solve_ivp
event_thresholds = [crash_event, skip_event]
